In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["GH_TOKEN"] = secrets.get_secret("GITHUB_SECRET")

In [ ]:

!git clone https://github.com/Vital-ar/adversarial_physics_problems_solver.git
%cd adversarial_physics_problems_solver

In [ ]:
!pip install -r requirements/kaggle.txt
!pip install -e .

In [ ]:
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=True)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/aligntune_v3

In [ ]:
!python scripts/05_train_aligntune.py --dataset verifier_e4_curated --config configs/training/aligntune_training_v3.yaml --push-to-hub


In [ ]:
!python scripts/05_train_aligntune.py --dataset solver_e3_canonical_llm --config configs/training/aligntune_training_v3.yaml --push-to-hub


In [ ]:
!python scripts/06_eval_solver.py --experiment E3_v3 --adapter-dir models/adapters/solver_e3_canonical_llm_v3_lora/final_adapter --raw-prompt --validation-path data/sft_v3/solver_e3_canonical_llm/validation.jsonl --test-path data/sft_v3/solver_e3_canonical_llm/test.jsonl --output-dir results/eval_e3_v3


In [ ]:
!python scripts/05_train_aligntune.py --dataset solver_e1_manual_det --config configs/training/aligntune_training_v3.yaml --push-to-hub


In [ ]:
!python scripts/06_eval_solver.py --experiment E1_v3 --adapter-dir models/adapters/solver_e1_manual_det_v3_lora/final_adapter --raw-prompt --validation-path data/sft_v3/solver_e1_manual_det/validation.jsonl --test-path data/sft_v3/solver_e1_manual_det/test.jsonl --output-dir results/eval_e1_v3


In [ ]:
%%bash
EXP_NAME="e3-e4v-e1"   # change to e2, e3, e4a, etc. in other notebooks

git config --global user.name "vital-ar"
git config --global user.email "vitalii.lvaryk.2024@gmail.com"

git checkout -B "train-v3-${EXP_NAME}"

git remote set-url origin https://${GH_TOKEN}@github.com/Vital-ar/adversarial_physics_problems_solver.git
git add models/ runs/ configs/ results/
git add .
git commit -m "kaggle :: add training v3 run outputs for ${EXP_NAME}" || true
git push -u origin "train-v3-${EXP_NAME}" --force